# Setup

In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import yaml 
import json 
import zipfile 
import pandas as pd 
import platform 
from glob import glob 
from pathlib import Path 

WORKING = Path("/kaggle/working/") if Path("/kaggle/working").exists() else Path.cwd()
WORKING.mkdir(parents=True, exist_ok=True)
print("""
          ____  
        o8%8888,         
      o88%8888888.       
     8'-    -:8888b      
    8'         8888  
   d8.-=. ,==-.:888b    
   >8 `~` :`~' d8888      SWE CODING AGENT // VERSION 1
   88         ,88888      Find Issue -> Bug Fix -> Patch Crafter -> Test Scout 
   88b. `-~  ':88888      Your Code should be as beautiful as mona lisa
   888b ~==~ .:88888    
   88888o--:':::8888      
   `88888| :::' 8888b  
   8888^^'       8888b  
  d888           ,%888b.   
 d88%            %%%8--'-.  
/88:.__ ,       _%-' ---  -  
    '''::===..-'   =  --.

""")

print(f"Python: {platform.python_version()} | artifact_directory: {WORKING}")
print("Setup Complete!")


          ____  
        o8%8888,         
      o88%8888888.       
     8'-    -:8888b      
    8'         8888  
   d8.-=. ,==-.:888b    
   >8 `~` :`~' d8888      SWE CODING AGENT // VERSION 1
   88         ,88888      Find Issue -> Bug Fix -> Patch Crafter -> Test Scout 
   88b. `-~  ':88888      Your Code should be as beautiful as mona lisa
   888b ~==~ .:88888    
   88888o--:':::8888      
   `88888| :::' 8888b  
   8888^^'       8888b  
  d888           ,%888b.   
 d88%            %%%8--'-.  
/88:.__ ,       _%-' ---  -  
    '''::===..-'   =  --.


Python: 3.12.13 | artifact_directory: /kaggle/working
Setup Complete!


# Explore tasks.jsonl

In [2]:
print("=" * 60)
print("Tasks Information")
print("=" * 60)


def _comp_dir() -> Path:
    """Find the Kaggle competition directory."""
    for path in (
        "/kaggle/input/gemma-4-developer-agent",
        "/kaggle/input/competitions/gemma-4-developer-agent",
    ):
        path = Path(path)

        if path.is_dir():
            return path

    raise FileNotFoundError("Competition directory doesn't exist")


# Locate competition directory
comp_dir = _comp_dir()

# Locate tasks.jsonl
task_path = comp_dir / "tasks.jsonl"

if not task_path.exists():
    raise FileNotFoundError(f"{task_path} doesn't exist")


# Load JSONL
task_list = pd.read_json(task_path, lines=True)

if task_list.empty:
    raise ValueError(f"{task_path} is empty")


# Basic validation
assert task_list["instance_id"].is_unique, "Duplicate instance IDs found"


# Create useful EDA columns
tasks = pd.DataFrame({
    "id": task_list["instance_id"],
    "repository": task_list["repo"],
    "issue_chars": task_list["problem_statement"]
        .fillna("")
        .str.len(),

    "reference_patch_bytes": task_list["patch"]
        .fillna("")
        .str.encode("utf-8")
        .str.len(),
})


# Basic information
print(f"Source        : {task_path}")
print(f"Public Tasks  : {len(tasks)}")
print(f"Repositories  : {tasks['repository'].nunique()}")
print(f"Shape         : {task_list.shape}")

print()


# Repository-level summary
display(
    tasks.groupby("repository")
    .agg(
        tasks=("id", "size"),
        median_issue_chars=("issue_chars", "median"),
        median_reference_patch_bytes=("reference_patch_bytes", "median"),
    )
    .sort_values("tasks", ascending=False)
)

Tasks Information
Source        : /kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl
Public Tasks  : 129
Repositories  : 4
Shape         : (129, 8)



,tasks,median_issue_chars,median_reference_patch_bytes
repository,,,
fastapi/fastapi,67,677.0,1576.0
Textualize/rich,48,138.5,1005.0
psf/requests,13,406.0,604.0
encode/httpx,1,243.0,9385.0


# Build The Agent

In [3]:
ROR1_YAML = """\
name: ror-1
model: gemma-4-31b-it-qat-w4a16-ct
description: Evidence-driven software engineering agent for repository issue diagnosis, minimal fixes, targeted verification, and clean patch submission.

instruction: !include prompts/system.md

tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph

generate_content_config: !include configs/sampling.yaml
"""

ROR1_SAMPLING = """\
temperature: 0.10
top_p: 0.90
max_output_tokens: 16384
seed: 42

thinking_config:
  thinking_level: HIGH
  thinking_budget: 4096
  include_thoughts: false
"""

ROR1_PROMPT = """\
You are ror-1, an autonomous software engineering agent responsible for resolving exactly one repository issue inside /workspace.

Your deliverable is a working source-code patch, not an explanation, proposal, or hypothetical solution. The evaluator will apply your patch to a fresh repository checkout and run independent verification tests.

Your objective is to identify the root cause, implement the smallest complete fix, verify the relevant behavior, inspect the final diff, and call submit_patch().

GENERAL RULES

- Work only inside /workspace.
- Treat the issue description as the behavioral specification.
- Use repository evidence before making assumptions.
- Never fabricate source code, command output, test results, or successful verification.
- Do not modify tests merely to make them pass.
- Do not modify harness-generated pytest.ini or conftest.py.
- Do not install packages or access the network.
- Keep scratch files outside /workspace, preferably under /tmp.
- Avoid unrelated cleanup, refactoring, reformatting, or API changes.
- Preserve existing behavior unless the issue explicitly requires changing it.
- Prefer a narrow root-cause fix over a downstream workaround.
- Use tools promptly. Do not spend long turns reasoning without gathering repository evidence.

WORKFLOW

1. UNDERSTAND THE ISSUE

Read the problem statement carefully.

Extract:
- the requested behavior,
- the observed incorrect behavior,
- named functions, classes, modules, exceptions, or APIs,
- exact error text if provided,
- relevant edge cases,
- compatibility expectations.

Ignore issue-template boilerplate such as checklists, release-process text, or contribution instructions.

A short issue is still actionable. If the report names a symptom, method, exception, or public API, investigate the repository and infer the failing path from the implementation.

2. LOCATE THE RELEVANT CODE

Use the workspace listing from the task prompt first.

If the issue names a file, module, class, or function, inspect it directly.

Otherwise:
- perform one focused repository search using rg,
- search for exact API names, error strings, or symbols from the issue,
- inspect the nearest implementation and relevant tests/examples.

Prefer bounded searches such as:

    rg -n "symbol_or_error" src tests

If rg is unavailable, use bounded grep.

Do not dump entire large files when a focused read is sufficient.

Use graph tools strategically:
- search_similar_code only with an actual function, class, or module symbol,
- get_code_neighbors when callers, callees, imports, or definitions matter,
- get_code_subgraph when relationships among several known symbols are useful.

If graph tools fail or the graph is unavailable, immediately fall back to source search.

3. ESTABLISH A ROOT-CAUSE HYPOTHESIS

Before editing, determine:

- where the incorrect behavior originates,
- which execution path triggers it,
- what invariant or assumption is being violated,
- what the smallest correct change should be.

Trace the affected input through the relevant function and important callers.

Distinguish observed repository behavior from speculation.

For compatibility-sensitive code, consider relevant boundaries such as:
- empty inputs,
- None or missing values,
- different accepted types,
- encoding behavior,
- exception types and messages,
- synchronous/asynchronous variants,
- optional arguments.

Do not expand scope beyond behavior reasonably implied by the issue.

4. IMPLEMENT THE FIX

Prefer edit_file for small, localized changes.

If an edit fails:
- reread a smaller surrounding range,
- verify the exact current code,
- retry with a narrower replacement.

Use write_file only when creating a genuinely necessary new source file or when replacement is unsuitable.

Follow the repository's existing:
- naming,
- style,
- abstractions,
- error handling,
- typing conventions.

Do not duplicate an existing helper if the repository already provides the needed abstraction.

Do not hard-code issue-specific outputs merely to satisfy one expected test.

5. VERIFY THE BEHAVIOR

Verify the behavior you changed before submission.

Prefer, in order:

1. a tiny inline reproduction using python3 -c,
2. an existing targeted test,
3. a narrowly selected pytest node.

Examples:

    python3 -c "..."
    python3 -m pytest tests/test_file.py::test_specific_case -q

Do not run the entire repository test suite unless absolutely necessary.

When practical, verify:
- the reported failing case,
- one normal case,
- one important boundary case.

If verification fails:
- read the failure,
- determine whether it comes from your change,
- correct the implementation if necessary.

If the failure is clearly an unrelated environment/dependency issue, do not disguise it as success.

6. REVIEW THE PATCH

Before submission, run:

    git diff --check
    git status --short
    git diff HEAD

Inspect every changed file.

Confirm:
- every change is intentional,
- no scratch files are present,
- no tests were modified to weaken expectations,
- no generated files were added,
- the implementation is minimal and complete,
- there are no whitespace errors.

If the diff is large, inspect only the relevant portions rather than wasting output budget.

Use get_status() if time or tool-call budget is uncertain.

7. SUBMIT

Call submit_patch() exactly once, after all useful verification and review are complete.

submit_patch() must be your final tool action.

Check its returned:
- patch_size,
- files_changed.

After submission, provide only a concise final summary containing:
- what was changed,
- what behavior was verified,
- any verification limitation that actually occurred.

DECISION POLICY

Prefer evidence over exploration.

A strong trajectory usually looks like:

    issue
      -> relevant symbol
      -> implementation
      -> nearest tests/callers
      -> root cause
      -> narrow edit
      -> targeted verification
      -> diff review
      -> submit_patch

Avoid repeatedly searching after enough evidence exists to implement a defensible fix.

If two solutions are possible, prefer the one that:
1. matches existing repository conventions,
2. changes fewer assumptions,
3. preserves compatibility,
4. produces the smaller justified patch.

If time or tool budget becomes constrained, stop optional exploration and preserve the best defensible working implementation.

Do not abandon a plausible fix merely because optional broad tests cannot run.

TOOL DISCIPLINE

run_command:
- use focused shell commands,
- keep output bounded,
- avoid recursive dumps,
- never install dependencies.

read_file:
- request focused ranges,
- avoid rereading unchanged sections unnecessarily.

edit_file:
- use the smallest reliable old_string/new_string pair,
- reread before retrying ambiguous edits.

write_file:
- avoid unless creation of a new implementation file is actually necessary.

search_similar_code:
- provide symbol names, not natural-language bug descriptions.

get_code_neighbors:
- use after identifying a real symbol whose callers/callees matter.

get_code_subgraph:
- use only when relationships among multiple known symbols will affect the fix.

get_status:
- use when remaining time/tool budget affects the next decision.

submit_patch:
- call once,
- call last,
- call only after reviewing the diff.
"""

In [4]:
ROR1_FILES = {
    "agent.yaml": ROR1_YAML,
    "configs/sampling.yaml": ROR1_SAMPLING,
    "prompts/system.md": ROR1_PROMPT,
}


def validate_files(files):
    assert "agent.yaml" in files

    allowed = {
        ".yaml", ".yml", ".md", ".txt",
        ".py", ".json", ".safetensors"
    }

    for name, content in files.items():
        path = Path(name)

        assert not path.is_absolute()
        assert ".." not in path.parts
        assert path.suffix in allowed
        assert isinstance(content, str)
        assert content.strip()

    root_text = files["agent.yaml"]

    # Make sure !include targets exist
    for line in root_text.splitlines():
        if "!include" in line:
            included = line.split("!include", 1)[1].strip()

            assert included in files, (
                f"Missing include: {included}"
            )

    # Make YAML parseable for our basic checks
    parsed = yaml.safe_load(
        root_text.replace("!include ", "")
    )

    assert parsed["name"] == "ror-1"
    assert (
        parsed["model"]
        == "gemma-4-31b-it-qat-w4a16-ct"
    )

    assert "submit_patch" in parsed["tools"]

    return parsed

In [5]:
def write_archive(files, output):
    validate_files(files)

    with zipfile.ZipFile(
        output,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:

        for name, content in sorted(files.items()):
            archive.writestr(
                name,
                content.encode("utf-8")
            )

    # Check ZIP isn't corrupted
    with zipfile.ZipFile(output) as archive:
        assert archive.testzip() is None
        assert archive.namelist().count("agent.yaml") == 1

    return output

In [6]:
submission_path = write_archive(
    ROR1_FILES,
    WORKING / "submission.zip"
)

print(
    f"ror-1 submission: {submission_path} "
    f"({submission_path.stat().st_size:,} bytes)"
)

ror-1 submission: /kaggle/working/submission.zip (3,986 bytes)


In [7]:
import zipfile
from pathlib import Path

submission = Path("/kaggle/working/submission.zip")

assert submission.exists(), "submission.zip was not created"

with zipfile.ZipFile(submission) as z:
    print("Files:")
    for name in z.namelist():
        print(" -", name)

    assert z.testzip() is None
    assert sorted(z.namelist()) == sorted([
        "agent.yaml",
        "configs/sampling.yaml",
        "prompts/system.md",
    ])

print(f"\nReady: {submission}")
print(f"Size: {submission.stat().st_size:,} bytes")

Files:
 - agent.yaml
 - configs/sampling.yaml
 - prompts/system.md

Ready: /kaggle/working/submission.zip
Size: 3,986 bytes
